# Atelier 1 — Notebook 1 : Exploration du Cost and Usage Report

**Objectif** : prendre en main un fichier de facturation cloud (format CUR simplifié) et en extraire les premières informations utiles.

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '${:,.2f}'.format)

## Cellule 1 — Charger le dataset

In [ ]:
CUR_PATH = Path('../../ressources/datasets/cur_sample.csv')
df = pd.read_csv(CUR_PATH, parse_dates=['usage_date'])
print(f'Lignes : {len(df):,}')
print(f'Période : {df.usage_date.min()} → {df.usage_date.max()}')
df.head()

## Cellule 2 — Vue d'ensemble

Calculer le coût total, le coût moyen journalier, le nombre de services et de comptes.

In [ ]:
print(f"Coût total : ${df.unblended_cost.sum():,.2f}")
print(f"Coût moyen / jour : ${df.groupby('usage_date').unblended_cost.sum().mean():,.2f}")
print(f"Nombre de services : {df.service.nunique()}")
print(f"Nombre de comptes : {df.account_id.nunique()}")
print(f"Nombre d'équipes (taggées) : {df.tag_team.replace('', pd.NA).nunique()}")

## Cellule 3 — Top services par coût

In [ ]:
top_services = (
    df.groupby('service').unblended_cost.sum()
      .sort_values(ascending=False)
      .head(10)
)
fig = px.bar(
    top_services.reset_index(),
    x='service', y='unblended_cost',
    title='Top 10 services par coût (90 jours)',
    labels={'unblended_cost': 'Coût ($)', 'service': 'Service AWS'}
)
fig.show()

## Cellule 4 — Tendance journalière

In [ ]:
daily = df.groupby('usage_date').unblended_cost.sum().reset_index()
fig = px.line(daily, x='usage_date', y='unblended_cost', 
              title='Coût journalier total sur 90 jours',
              labels={'unblended_cost': 'Coût ($)', 'usage_date': 'Date'})
fig.show()

👉 **Q1 (exo 2)** : repérez-vous un pic ? À quelle date ? Quelle hypothèse ?

## Cellule 5 — Décomposition par compte

In [ ]:
account_labels = {
    '111111111111': 'prod',
    '222222222222': 'staging',
    '333333333333': 'dev',
    '444444444444': 'sandbox',
}
df['account_name'] = df['account_id'].map(account_labels)
by_account = df.groupby('account_name').unblended_cost.sum().reset_index()
fig = px.pie(by_account, values='unblended_cost', names='account_name',
             title='Répartition du coût par compte')
fig.show()

## Cellule 6 — Heatmap service × compte

In [ ]:
pivot = df.pivot_table(
    index='service', columns='account_name',
    values='unblended_cost', aggfunc='sum', fill_value=0
)
fig = px.imshow(pivot, text_auto='.0f', aspect='auto',
                title='Coût par service × compte ($)',
                color_continuous_scale='Reds')
fig.show()

## Cellule 7 — Évolution week-end vs semaine (env non-prod)

In [ ]:
df['is_weekend'] = df['usage_date'].dt.dayofweek >= 5
non_prod = df[df['account_name'].isin(['dev', 'sandbox'])]
we_stats = non_prod.groupby(['account_name', 'is_weekend']).unblended_cost.mean().reset_index()
we_stats['period'] = we_stats['is_weekend'].map({True: 'Week-end', False: 'Semaine'})
fig = px.bar(we_stats, x='account_name', y='unblended_cost', color='period',
             barmode='group', title='Coût moyen ligne CUR : semaine vs week-end')
fig.show()

👉 **Observation** : un environnement dev/sandbox correctement géré devrait montrer une **baisse week-end**. Si la barre week-end ≈ semaine, c'est un **gisement d'économie**.

## Cellules 8 à 12 — À vous !

Compléter avec :
8. Répartition par région
9. Évolution de S3 dans le temps (croissance attendue)
10. Top 5 `usage_type` toutes catégories confondues
11. Décomposition par équipe (en filtrant les non-taggés)
12. Synthèse en chiffres clés à mettre dans le rapport